# FinPilot AI — Hackathon Notebook

Notebook portátil para treinar e demonstrar o núcleo do **FinPilot AI** em um ambiente Google/Colab/Workbench.

## Objetivo
- Ler um extrato financeiro em CSV
- Calcular indicadores financeiros
- Simular compras com **Safe Spend**
- Detectar gastos recorrentes
- Projetar saldo até o fim do mês
- Detectar gastos fora do padrão
- Integrar com Gemini de forma opcional
- Manter os cálculos no Python e usar a IA apenas para interpretação

> Os dados de exemplo são fictícios. Não use credenciais reais diretamente no notebook.


## 1. Instalação

No Colab/Workbench, rode a célula abaixo. Se o ambiente já tiver as bibliotecas, ela apenas confirmará/atualizará o necessário.


In [ ]:
!pip -q install pandas numpy google-genai python-dotenv

## 2. Imports e configuração

In [ ]:
import os
import json
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 3. Carregar dados

Você pode:
1. usar o CSV fictício criado abaixo; ou
2. fazer upload de um CSV próprio com as colunas:

`date, description, category, type, amount`

Valores esperados em `type`: `entrada` ou `saida`.


In [ ]:
sample_data = [
    ["2026-09-01", "Salario", "Receita", "entrada", 5800],
    ["2026-09-02", "Aluguel", "Moradia", "saida", 1500],
    ["2026-09-03", "Supermercado", "Alimentacao", "saida", 412],
    ["2026-09-04", "Uber", "Transporte", "saida", 48],
    ["2026-09-05", "Netflix", "Assinaturas", "saida", 59],
    ["2026-09-06", "Restaurante", "Alimentacao", "saida", 120],
    ["2026-09-07", "Internet", "Contas", "saida", 110],
    ["2026-09-08", "Energia", "Contas", "saida", 185],
    ["2026-09-09", "Farmacia", "Saude", "saida", 75],
    ["2026-09-10", "Academia", "Saude", "saida", 120],
    ["2026-09-11", "Uber", "Transporte", "saida", 35],
    ["2026-09-12", "Shopping", "Lazer", "saida", 250],
    ["2026-09-13", "Supermercado", "Alimentacao", "saida", 280],
    ["2026-09-14", "Spotify", "Assinaturas", "saida", 22],
    ["2026-09-15", "Freelance", "Receita", "entrada", 600],
    ["2026-09-16", "Restaurante", "Alimentacao", "saida", 98],
    ["2026-09-17", "Gasolina", "Transporte", "saida", 180],
    ["2026-09-18", "Cinema", "Lazer", "saida", 70],
    ["2026-09-19", "Supermercado", "Alimentacao", "saida", 330],
    ["2026-09-20", "Telefone", "Contas", "saida", 65],
    ["2026-09-21", "Uber", "Transporte", "saida", 250],  # anomalia proposital
]

df = pd.DataFrame(
    sample_data,
    columns=["date", "description", "category", "type", "amount"]
)

df["date"] = pd.to_datetime(df["date"])
df["amount"] = pd.to_numeric(df["amount"])
df["type"] = df["type"].str.strip().str.lower()

df.head()


### Upload opcional no Google Colab

Descomente a célula abaixo se estiver no Colab e quiser enviar um CSV.


In [ ]:
# from google.colab import files
# uploaded = files.upload()
# filename = next(iter(uploaded))
# df = pd.read_csv(filename)
# df["date"] = pd.to_datetime(df["date"])
# df["amount"] = pd.to_numeric(df["amount"])
# df["type"] = df["type"].astype(str).str.strip().str.lower()
# df.head()


## 4. Validação do DataFrame

In [ ]:
def validate_dataframe(df):
    required_columns = {"date", "description", "category", "type", "amount"}
    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Colunas ausentes: {sorted(missing)}")

    invalid_types = set(df["type"].dropna().astype(str).str.lower()) - {"entrada", "saida"}
    if invalid_types:
        raise ValueError(f"Valores inválidos em 'type': {sorted(invalid_types)}")

    pd.to_datetime(df["date"])
    pd.to_numeric(df["amount"])

    return True

validate_dataframe(df)


## 5. Motor financeiro determinístico

In [ ]:
def calculate_total_income(df):
    return float(df.loc[df["type"] == "entrada", "amount"].sum())

def calculate_total_expenses(df):
    return float(df.loc[df["type"] == "saida", "amount"].sum())

def calculate_balance(df):
    return calculate_total_income(df) - calculate_total_expenses(df)

def expenses_by_category(df):
    return (
        df.loc[df["type"] == "saida"]
        .groupby("category")["amount"]
        .sum()
        .sort_values(ascending=False)
    )

def calculate_safe_spend(df, reserve_percentage=0.10):
    income = calculate_total_income(df)
    expenses = calculate_total_expenses(df)
    balance = income - expenses
    reserve = income * reserve_percentage
    safe_spend = max(balance - reserve, 0)

    return {
        "income": round(income, 2),
        "expenses": round(expenses, 2),
        "balance": round(balance, 2),
        "reserve": round(reserve, 2),
        "safe_spend": round(safe_spend, 2),
    }

summary = calculate_safe_spend(df)
summary


## 6. Safe Spend — impacto de uma compra

In [ ]:
def calculate_purchase_impact(df, purchase_amount):
    purchase_amount = float(purchase_amount)
    safe_data = calculate_safe_spend(df)

    balance = float(safe_data["balance"])
    safe_spend = float(safe_data["safe_spend"])
    reserve = float(safe_data["reserve"])

    remaining_balance = balance - purchase_amount
    remaining_safe_spend = safe_spend - purchase_amount
    usage_percentage = (purchase_amount / safe_spend * 100) if safe_spend > 0 else 0.0

    if purchase_amount > safe_spend:
        risk = "alto"
    elif usage_percentage >= 75:
        risk = "moderado"
    else:
        risk = "baixo"

    excess_amount = max(purchase_amount - safe_spend, 0.0)

    return {
        "purchase_amount": round(purchase_amount, 2),
        "current_balance": round(balance, 2),
        "reserve": round(reserve, 2),
        "safe_spend": round(safe_spend, 2),
        "remaining_balance": round(remaining_balance, 2),
        "remaining_safe_spend": round(remaining_safe_spend, 2),
        "usage_percentage": round(usage_percentage, 2),
        "excess_amount": round(excess_amount, 2),
        "risk": risk,
    }

calculate_purchase_impact(df, 700)


## 7. Gastos recorrentes

In [ ]:
def detect_recurring_expenses(df):
    expenses = df[df["type"] == "saida"].copy()

    recurring = (
        expenses
        .groupby(["description", "category"])["amount"]
        .agg(["count", "sum", "mean"])
        .reset_index()
    )

    recurring = recurring[recurring["count"] >= 2].sort_values("sum", ascending=False)

    return [
        {
            "description": row["description"],
            "category": row["category"],
            "occurrences": int(row["count"]),
            "total_amount": round(float(row["sum"]), 2),
            "average_amount": round(float(row["mean"]), 2),
        }
        for _, row in recurring.iterrows()
    ]

detect_recurring_expenses(df)


## 8. Forecast de saldo até o fim do mês

In [ ]:
def forecast_month_end_balance(df):
    working_df = df.copy()
    working_df["date"] = pd.to_datetime(working_df["date"])

    expenses = working_df[working_df["type"] == "saida"].copy()
    income = working_df.loc[working_df["type"] == "entrada", "amount"].sum()
    total_expenses = expenses["amount"].sum()
    current_balance = income - total_expenses

    if working_df.empty:
        return {
            "current_balance": 0.0,
            "average_daily_expense": 0.0,
            "days_observed": 0,
            "days_remaining": 0,
            "projected_remaining_expenses": 0.0,
            "projected_month_end_balance": 0.0,
        }

    first_date = working_df["date"].min()
    last_date = working_df["date"].max()

    days_observed = max((last_date - first_date).days + 1, 1)
    average_daily_expense = total_expenses / days_observed

    month_end = last_date + pd.offsets.MonthEnd(0)
    days_remaining = max((month_end - last_date).days, 0)

    projected_remaining_expenses = average_daily_expense * days_remaining
    projected_month_end_balance = current_balance - projected_remaining_expenses

    return {
        "current_balance": round(float(current_balance), 2),
        "average_daily_expense": round(float(average_daily_expense), 2),
        "days_observed": int(days_observed),
        "days_remaining": int(days_remaining),
        "projected_remaining_expenses": round(float(projected_remaining_expenses), 2),
        "projected_month_end_balance": round(float(projected_month_end_balance), 2),
    }

forecast_month_end_balance(df)


## 9. Detecção simples de anomalias

In [ ]:
def detect_spending_anomalies(df, threshold_multiplier=2.0):
    expenses = df[df["type"] == "saida"].copy()
    results = []

    for (description, category), group in expenses.groupby(["description", "category"]):
        if len(group) < 2:
            continue

        average_amount = group["amount"].mean()
        threshold = average_amount * threshold_multiplier

        anomalies = group[group["amount"] > threshold]

        for _, row in anomalies.iterrows():
            results.append({
                "date": str(row["date"]),
                "description": description,
                "category": category,
                "amount": round(float(row["amount"]), 2),
                "average_amount": round(float(average_amount), 2),
                "threshold": round(float(threshold), 2),
                "difference_from_average": round(float(row["amount"] - average_amount), 2),
            })

    return results

detect_spending_anomalies(df)


## 10. Resumo de demonstração

Esta célula consolida os principais resultados para uma demo rápida.


In [ ]:
demo = {
    "summary": calculate_safe_spend(df),
    "top_category": expenses_by_category(df).head(1).to_dict(),
    "purchase_700": calculate_purchase_impact(df, 700),
    "recurring": detect_recurring_expenses(df),
    "forecast": forecast_month_end_balance(df),
    "anomalies": detect_spending_anomalies(df),
}

print(json.dumps(demo, indent=2, ensure_ascii=False, default=str))


## 11. Gemini — integração opcional

Use **somente se o ambiente do evento permitir**.

### Opção A — API key
Defina a variável de ambiente `GEMINI_API_KEY`.

No Colab, prefira usar *Secrets* em vez de escrever a chave no código.

### Opção B — credencial/projeto fornecido pelo evento
Se o Itaú/Google fornecer um projeto ou credenciais próprias, substitua apenas a camada de autenticação. O motor financeiro acima permanece igual.


In [ ]:
# Integração opcional
# Não execute sem configurar credenciais.

from google import genai

def get_gemini_client():
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError(
            "GEMINI_API_KEY não configurada. "
            "Use o mecanismo de credenciais disponibilizado pelo evento."
        )
    return genai.Client(api_key=api_key)

def ask_gemini_with_context(question, df):
    client = get_gemini_client()

    context = {
        "financial_summary": calculate_safe_spend(df),
        "expenses_by_category": expenses_by_category(df).to_dict(),
        "forecast": forecast_month_end_balance(df),
        "recurring_expenses": detect_recurring_expenses(df),
        "anomalies": detect_spending_anomalies(df),
    }

    prompt = f'''
Você é o FinPilot AI.

Use somente os dados calculados abaixo.
Não invente valores.
Os cálculos foram feitos em Python.

DADOS:
{json.dumps(context, ensure_ascii=False, default=str)}

PERGUNTA:
{question}

Responda em português do Brasil, de forma objetiva.
'''

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
    )

    return response.text

# Exemplo:
# print(ask_gemini_with_context("Posso gastar R$ 700 hoje?", df))


## 12. Checklist para o evento

Antes da competição:

- [ ] Repositório GitHub atualizado
- [ ] Notebook salvo localmente
- [ ] ZIP do projeto no notebook
- [ ] `requirements.txt` atualizado
- [ ] CSV fictício de teste
- [ ] Testar execução sem Gemini
- [ ] Confirmar método de autenticação Google fornecido pelo evento
- [ ] Não expor API keys
- [ ] Validar um fluxo de demo de 2–5 minutos

### Fluxo de demo sugerido

1. Mostrar o extrato
2. Mostrar resumo financeiro
3. Perguntar: **"Posso gastar R$ 700?"**
4. Mostrar cálculo determinístico
5. Mostrar forecast
6. Mostrar anomalia
7. Explicar que a IA interpreta resultados produzidos por ferramentas Python
